Применяется OLS - ordinary least squares - метод наименьших квадратов. Это способ подобрать коэффициенты в линейной регрессии так, чтобы сумма квадратов ошибок была минимальной.

Но в моих данных есть категориальные признаки, а метод OLS не принимает текст, только числа. 
Мы сделаем baseline - OLS только на числовых признаках. Baseline- это простая модель, с которой мы будем сравнивать все последующие модели, доабвляя наши улучшения.

In [6]:
import pandas as pd
import os
from pathlib import Path
import statsmodels.api as sm
p = Path.cwd()
while not (p / 'data').exists() and p != p.parent:
    p = p.parent
os.chdir(p)
print('рабочая папка:', os.getcwd())

рабочая папка: c:\Users\sea\Desktop\---\housing-price-econometrics


In [2]:
#загружаем данные
df = pd.read_csv('data/interim/ames_cleaned.csv', encoding='latin1')

In [8]:
#отбираем числовые признаки
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(numeric_cols)

['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold', 'SalePrice']


In [11]:
#убираем целевую колонку SalePrice 
X = df.select_dtypes(include=['int64', 'float64']).drop(columns=['SalePrice', 'Order', 'PID'])
y=df['SalePrice']

In [12]:
X=sm.add_constant(X)
model=sm.OLS(y,X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.837
Method:                 Least Squares   F-statistic:                     444.1
Date:                Thu, 08 Oct 2026   Prob (F-statistic):               0.00
Time:                        19:10:53   Log-Likelihood:                -34555.
No. Observations:                2930   AIC:                         6.918e+04
Df Residuals:                    2895   BIC:                         6.939e+04
Df Model:                          34                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            8.783e+05    9.3e+05     

C:\Users\sea\AppData\Local\Temp\ipykernel_2788\3558253254.py:2: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model=sm.OLS(y,X).fit()


Модель OLS на 34 числовых признаках показала:
R2=0.839 это коэффициент дереминации, означает, что модель объясняет 83.9% вариации SalePrice. Остальные 16.1% — шум или признаки, которые мы не учли.
По p-value мы можем оценить значимость коэффициента на модель. (Если >0.05 -коэф. статически не значим, если <0,05 - значим).  
В наших данных есть как значимые, так и нет. Не значимые признаки не дают статически большого вклада и в дальшейших моделях их можно рассмотреть к удалению.
F-statistic=444.1, Prob (F-statistic)=0 - это значит что модель в целом значима. (Это говорит о том, что один из факторов существенно влияет на зависимую переменную)
"The smallest eigenvalue is 3.62e-21." - также у нас присутствует знак мультиколлинеарности.
По Jarque-Bera была проверена нулевая гипотеза согласна которой остатки распределены нормально, но поскольку p-значение равно 0(<0.05), это говорит о том, что гипотеза о нормальности отвергается. Остатки сильно отклоняются от нормального закона.
Значение Ассиметрии=-1,182 (левосторонняя ассиметрия) это говорит о том, что левых хвост распределения ошибок длиннее. 
Значение Эксцесса=46,250 (положительный эксцесс) это говорит о том, что пик высокий и острый, а хвосты тяжелые. т.е. в данных встречаются резкие скачки, аномалии и экстримальные значения.
Критерий Дарбина= 1,516. Он лежит в нормальном диапазоне, что говорит о нормальной автокорреляции. 
Cond. No. = 1.20e+16 говорит о критической мультиколлинеарности в наших данных
